In [104]:
############################################################

# EECE 5644 - Machine Learning
# Skylar Denno
# July 22, 2026

# ASSIGNMENT 6
# Long Short-Term Memory RNNs

############################################################

In [105]:
import torch
import torch.nn as nn
import pandas as pd
import numpy as np
from torch.utils.data import TensorDataset, DataLoader

# pick the best avaliable device for training
if torch.backends.mps.is_available():
    device = torch.device("mps")
elif torch.cuda.is_available():
    device = torch.device("cuda")
else:
    device = torch.device("cpu")

df = pd.read_csv('household_power_consumption.txt', sep=';', na_values='?')
print(df.head())

         Date      Time  Global_active_power  Global_reactive_power  Voltage  \
0  16/12/2006  17:24:00                4.216                  0.418   234.84   
1  16/12/2006  17:25:00                5.360                  0.436   233.63   
2  16/12/2006  17:26:00                5.374                  0.498   233.29   
3  16/12/2006  17:27:00                5.388                  0.502   233.74   
4  16/12/2006  17:28:00                3.666                  0.528   235.68   

   Global_intensity  Sub_metering_1  Sub_metering_2  Sub_metering_3  
0              18.4             0.0             1.0            17.0  
1              23.0             0.0             1.0            16.0  
2              23.0             0.0             2.0            17.0  
3              23.0             0.0             1.0            17.0  
4              15.8             0.0             1.0            17.0  


In [106]:
######################
# DATA PREPROCESSING #
######################

#missing_idx = df.index[df.isna().any(axis=1)][0]
#print(df.iloc[missing_idx - 3:missing_idx + 4])

# Justification: For this problem, interpolation was chosen, as it will attempt to preserve the
# local trends seen in the data. This power data varies temporally, in short-term and long-term patterns.
# Additionally, the minute long sampling period makes the data change smoothly, and using forward or back fill 
# would add unnatural jumps in the data.

# add an absolute time index so interpolation follows the chronological order
df['DateTime'] = pd.to_datetime(df['Date'] + ' ' + df['Time'], dayfirst=True)
df = df.drop(columns=['Date', 'Time']).set_index('DateTime').sort_index()
numeric_cols = df.columns

missing_count = int(df[numeric_cols].isna().sum().sum())
print("Number of missing values found.", missing_count)

# interpolate gaps from neighboring observations
df[numeric_cols] = df[numeric_cols].interpolate(method='time', limit_direction='both')

# fill ends of gaps (only) with forward and back fill
df[numeric_cols] = df[numeric_cols].ffill().bfill()

missing_after = int(df[numeric_cols].isna().sum().sum())
print("Number of missing values filled by interpolation.", missing_after)

#print(df.iloc[missing_idx - 3:missing_idx + 4])




Number of missing values found. 181853
Number of missing values filled by interpolation. 0


In [107]:
# Justification: Mean is used for volts, amps, power (kW) since they are comtinuously sampled,
# and the mean will represent the typical level over the hour.
# Summing is used for sub-metering channels because they represent
# ENERGY usage contributions in watt hours, so minute-to-minute totals add up.

agg_map = {
    'Global_active_power': 'mean',
    'Global_reactive_power': 'mean',
    'Voltage': 'mean',
    'Global_intensity': 'mean',
    'Sub_metering_1': 'sum',
    'Sub_metering_2': 'sum',
    'Sub_metering_3': 'sum'
}

df_h = df.resample('h').agg(agg_map)

print(df_h.head())

# separate features and target
feature_cols = df_h.columns.tolist()
target_col = 'Global_active_power'
target_idx = feature_cols.index(target_col)
values = df_h[feature_cols].to_numpy(dtype=np.float32)

                     Global_active_power  Global_reactive_power     Voltage  \
DateTime                                                                      
2006-12-16 17:00:00             4.222889               0.229000  234.643889   
2006-12-16 18:00:00             3.632200               0.080033  234.580167   
2006-12-16 19:00:00             3.400233               0.085233  233.232500   
2006-12-16 20:00:00             3.268567               0.075100  234.071500   
2006-12-16 21:00:00             3.056467               0.076667  237.158667   

                     Global_intensity  Sub_metering_1  Sub_metering_2  \
DateTime                                                                
2006-12-16 17:00:00         18.100000             0.0            19.0   
2006-12-16 18:00:00         15.600000             0.0           403.0   
2006-12-16 19:00:00         14.503333             0.0            86.0   
2006-12-16 20:00:00         13.916667             0.0             0.0   
2006-12-

In [108]:
##################
# MODEL TRAINING #
##################

target_start_times = []
X_list, y_list = [], []
for start in range(0, len(values) - 168 - 24 + 1, 4):
    x_window = values[start:start + 168, :]
    y_window = values[start + 168:start + 168 + 24, target_idx]

    X_list.append(x_window)
    y_list.append(y_window)
    target_start_times.append(df_h.index[start + 168])

X = np.array(X_list, dtype=np.float32)
y = np.array(y_list, dtype=np.float32)
target_start_times = pd.to_datetime(target_start_times)

In [109]:
# split off final 6 months for test set
cutoff_time = df_h.index.max() - pd.DateOffset(months=6)
test_mask = target_start_times >= cutoff_time
train_mask = target_start_times < cutoff_time

X_train_full, X_test = X[train_mask], X[test_mask]
y_train_full, y_test = y[train_mask], y[test_mask]

In [110]:
# create chronological validation split from the tail of the training data
val_frac = 0.2
val_size = max(1, int(len(X_train_full) * val_frac))
X_train, X_val = X_train_full[:-val_size], X_train_full[-val_size:]
y_train, y_val = y_train_full[:-val_size], y_train_full[-val_size:]

print(f'Train set: {len(X_train)}, validation set: {len(X_val)}, test set: {len(X_test)}')


Train set: 6001, validation set: 1500, test set: 1099


In [111]:
# Normalize using training statistics only.
feat_mean = X_train.reshape(-1, X_train.shape[-1]).mean(axis=0)
feat_std = X_train.reshape(-1, X_train.shape[-1]).std(axis=0) + 1e-8

X_train_n = (X_train - feat_mean) / feat_std
X_val_n = (X_val - feat_mean) / feat_std
X_test_n = (X_test - feat_mean) / feat_std

y_mean = y_train.mean()
y_std = y_train.std() + 1e-8

y_train_n = (y_train - y_mean) / y_std
y_val_n = (y_val - y_mean) / y_std
y_test_n = (y_test - y_mean) / y_std

batch_size = 64
train_ds = TensorDataset(torch.tensor(X_train_n), torch.tensor(y_train_n))
val_ds = TensorDataset(torch.tensor(X_val_n), torch.tensor(y_val_n))
test_ds = TensorDataset(torch.tensor(X_test_n), torch.tensor(y_test_n))

train_loader = DataLoader(train_ds, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_ds, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=batch_size, shuffle=False)

In [112]:
class LSTM_inst(nn.Module):
    def __init__(self, input_size, hidden_size=64, num_layers=2, forecast_horizon=24, dropout=0.2):
        super().__init__()
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout if num_layers > 1 else 0.0
        )
        self.head = nn.Linear(hidden_size, forecast_horizon)

    def forward(self, x):
        out, _ = self.lstm(x)
        last_hidden = out[:, -1, :]
        return self.head(last_hidden)

In [113]:
def calc_mse(model, loader, criterion):
    model.eval()
    total_loss = 0.0
    total_count = 0
    with torch.no_grad():
        for xb, yb in loader:
            xb = xb.to(device)
            yb = yb.to(device)
            pred = model(xb)
            loss = criterion(pred, yb)
            total_loss += loss.item() * xb.size(0)
            total_count += xb.size(0)
    return total_loss / max(total_count, 1)

In [114]:
#########################
# HYPERPARAMETER TUNING #
#########################

search_space = [
    {'hidden_size': 32, 'num_layers': 1, 'dropout': 0.0, 'lr': 1e-3},
    {'hidden_size': 64, 'num_layers': 1, 'dropout': 0.0, 'lr': 1e-3},
    {'hidden_size': 64, 'num_layers': 2, 'dropout': 0.2, 'lr': 1e-3},
    {'hidden_size': 128, 'num_layers': 2, 'dropout': 0.2, 'lr': 5e-4},
]
